In [99]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import numpy as np
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder,PowerTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import joblib
from xgboost import XGBRegressor


In [100]:
Train1=pd.read_csv("TRAIN DATA.csv")


In [101]:
Train1.shape

(1460, 81)

In [102]:
Train1.duplicated().sum()

np.int64(0)

In [103]:
Train1.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [104]:
Train1.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [105]:
Train1.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

In [107]:
#TARGET = "SalePrice"

X =Train1.drop(columns=["SalePrice"])
y =Train1["SalePrice"]
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [108]:
NUMERIC_FEATURES = x_train.select_dtypes(include=["number"]).columns.tolist()


In [109]:
CATEGORICAL_FEATURES = x_train.select_dtypes(include=["object"]).columns.tolist()

In [110]:
for col in NUMERIC_FEATURES:

    Q1 = x_train[col].quantile(0.25)
    Q3 = x_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    x_train[col]=(x_train[col].clip(lower,upper))
    x_test[col]=x_test[col].clip(lower,upper)

In [111]:
# ── 3. PREPROCESSING ──────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("Normalize",PowerTransformer())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

Transformer = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])


In [112]:
model=Pipeline([
    ("transform",Transformer),
    ("model",XGBRegressor(
    booster="gbtree",
    n_estimators=1000,
    learning_rate=0.1,
    reg_lambda=0.001,
    reg_alpha=0.001
))
])

In [41]:
#x_train=Transformer.fit_transform(x_train)

In [42]:
#x_test=Transformer.transform(x_test)

In [43]:
#x_test=pd.DataFrame(x_test,columns=Transformer.get_feature_names_out())

In [44]:
#x_train=pd.DataFrame(x_train,columns=Transformer.get_feature_names_out())

In [45]:
#x_train

In [104]:
#Transform = Pipeline([
  #  ("preprocessor", preprocessor)
    
  # ])


In [113]:
model.fit(x_train,y_train)

,steps,"[('transform', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [37]:
#y_train=np.log1p(y_train)

In [38]:
#y_test=np.log1p(y_test)

In [114]:
y_pred=model.predict(x_test)


In [33]:
#prediction2=np.expm1(y_pred)

In [34]:
#ytestorig=np.expm1(y_test)

In [35]:
#invpred=np.expm1(prediction2)

In [115]:
rmse = root_mean_squared_error(y_test,y_pred),
mae  = mean_absolute_error(y_test,y_pred),
r2   = r2_score(y_test, y_pred)

In [116]:
r2

0.9027420878410339

In [117]:
rmse

(27313.0,)

In [118]:
mae

(17006.001953125,)

In [119]:
import joblib as job
job.dump(model,"DATA_SIMPLE.joblib",compress=3)

['DATA_SIMPLE.joblib']

In [120]:
import os

print(os.path.getsize("DATA_SIMPLE.joblib"))

1030978


In [121]:
import os

print(os.path.exists("DATA_SIMPLE.joblib"))
print(os.path.getsize("DATA_SIMPLE.joblib"))

True
1030978


In [122]:
import os
print(os.getcwd())

C:\Users\DR SWETU\Videos\PYHON


In [123]:
x_test

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
892,893,20,RL,70.0,8414.0,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal
1105,1106,60,RL,98.0,12256.0,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
413,414,30,RM,56.0,8960.0,Pave,Grvl,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal
522,523,50,RM,50.0,5000.0,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal
1036,1037,20,RL,89.0,12898.0,Pave,NaN,IR1,HLS,AllPub,...,0,0,NaN,NaN,NaN,0,9,2009,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,480,30,RM,50.0,5925.0,Pave,NaN,Reg,Bnk,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2007,WD,Alloca
1361,1362,20,RL,111.5,16158.0,Pave,NaN,IR1,Low,AllPub,...,0,0,NaN,NaN,NaN,0,6,2009,WD,Normal
802,803,60,RL,63.0,8199.0,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2008,WD,Normal
651,652,70,RL,60.0,9084.0,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,10,2009,WD,Normal
